In [4]:
!pip install gradio

In [5]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import gradio as gr

## 1. Loading and cleaning data

In [6]:
df = pd.read_csv("Updated_sales.csv")
df = df.dropna(how='all')
df = df[df['Order Date'] != 'Order Date']

Convert Data Types for Feature Engineering

In [7]:
df['Price Each'] = pd.to_numeric(df['Price Each'])
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%m/%d/%y %H:%M', errors='coerce')

## 2. Evaluation Section (Train/Test Split)

We run an offline evaluation using a **Leave-One-Out** split. To improve our baseline accuracy, we apply three NLP Feature Engineering tricks to the text profiles:
1. **Chronological Sorting**: Respecting the timeline of user purchases.
2. **Price Personas**: Bucketing users by average spending habits (Cheap, Budget, Mid, Premium).
3. **Recency Weighting**: Duplicating the most recent purchase to artificially boost its TF-IDF weight.

In [8]:
print("\nRunning Offline Evaluation")
user_counts = df['Purchase Address'].value_counts()
valid_eval_users = user_counts[user_counts >= 3].index

eval_df = df[df['Purchase Address'].isin(valid_eval_users)].copy()

if len(eval_df) > 0:
    #Chronological Sorting
    eval_df = eval_df.sort_values(by=['Purchase Address', 'Order Date'])

    # Leave-One-Out Split
    test_df = eval_df.groupby('Purchase Address').tail(1)
    train_df = eval_df.drop(test_df.index)

    # Price Personas
    user_avg_price = train_df.groupby('Purchase Address')['Price Each'].mean()
    def get_persona(price):
        if price < 20: return "PERSONA_CHEAP"
        elif price < 100: return "PERSONA_BUDGET"
        elif price < 400: return "PERSONA_MID"
        else: return "PERSONA_PREMIUM"
    personas = user_avg_price.apply(get_persona)

    # Recency Weighting
    def create_doc(group):
        items = list(group['Product'])
        if len(items) > 0:
            items.append(items[-1]) # Duplicate the most recent item to boost its TF-IDF weight
        return ", ".join(items)

    train_matrix = train_df.groupby('Purchase Address').apply(create_doc).reset_index(name='Product')
    train_matrix['Persona'] = train_matrix['Purchase Address'].map(personas)

    # Combine products and persona tag into one string
    train_matrix['Combined_Features'] = train_matrix['Product'] + ", " + train_matrix['Persona']

    # TF-IDF Evaluation Model
    tfidf_eval = TfidfVectorizer()
    tfidf_matrix_eval = tfidf_eval.fit_transform(train_matrix['Combined_Features'])
    similarity_matrix_eval = cosine_similarity(tfidf_matrix_eval)

    test_matrix = test_df.groupby('Purchase Address')['Product'].apply(set).reset_index()

    def evaluate_recommender(k=5):
        precisions = []
        recalls = []

        for index, row in test_matrix.iterrows():
            target_address = row['Purchase Address']
            actual_hidden_purchases = row['Product']

            if target_address not in train_matrix['Purchase Address'].values:
                continue

            target_index = train_matrix[train_matrix['Purchase Address'] == target_address].index[0]

            # Fetch clean past purchases (without the persona or duplicated items)
            past_purchases = set(train_df[train_df['Purchase Address'] == target_address]['Product'].values)

            similar_customers = similarity_matrix_eval[target_index].argsort()[::-1][1:(k*5)+1]

            recommendations = []
            for cust_idx in similar_customers:
                cust_addr = train_matrix.iloc[cust_idx]['Purchase Address']
                cust_purchases = set(train_df[train_df['Purchase Address'] == cust_addr]['Product'].values)

                new_items = cust_purchases.difference(past_purchases)
                for item in new_items:
                    if item not in recommendations:
                        recommendations.append(item)

                if len(recommendations) >= k:
                    break

            recs = recommendations[:k]

            hits = set(recs).intersection(actual_hidden_purchases)
            precision = len(hits) / k
            recall = len(hits) / len(actual_hidden_purchases) if len(actual_hidden_purchases) > 0 else 0

            precisions.append(precision)
            recalls.append(recall)

        print(f"Metrics at K={k} (WITH Feature Engineering):")
        print(f"Mean Precision@{k}: {np.mean(precisions):.4f}")
        print(f"Mean Recall@{k}: {np.mean(recalls):.4f}\n")

    evaluate_recommender(k=5)


Running Offline Evaluation


/tmp/ipykernel_18140/1906373922.py:31: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_matrix = train_df.groupby('Purchase Address').apply(create_doc).reset_index(name='Product')


Metrics at K=5 (WITH Feature Engineering):
Mean Precision@5: 0.0560
Mean Recall@5: 0.2800



## 3. Production Model
Now that we've validated our mathematical approach, we apply the exact same feature engineering logic to **100% of the dataset** to build the final profile matrix. This ensures our User Interface has the most accurate behavioral data possible.

In [9]:
prod_df = df.sort_values(by=['Purchase Address', 'Order Date'])

prod_avg_price = prod_df.groupby('Purchase Address')['Price Each'].mean()
prod_personas = prod_avg_price.apply(get_persona)

def create_prod_doc(group):
    items = list(group['Product'])
    if len(items) > 0:
        items.append(items[-1])
    return ", ".join(items)

user_item_matrix = prod_df.groupby('Purchase Address').apply(create_prod_doc).reset_index(name='Product')
user_item_matrix['Persona'] = user_item_matrix['Purchase Address'].map(prod_personas)
user_item_matrix['Combined_Features'] = user_item_matrix['Product'] + ", " + user_item_matrix['Persona']

tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(user_item_matrix['Combined_Features'])
similarity_matrix = cosine_similarity(tfidf_matrix)

purchase_counts = prod_df['Purchase Address'].value_counts()
valid_customers = purchase_counts[purchase_counts > 1].index.tolist()
dropdown_choices = valid_customers[:1000]

print("Engine Ready! Launching UI\n")

/tmp/ipykernel_18140/2900132306.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  user_item_matrix = prod_df.groupby('Purchase Address').apply(create_prod_doc).reset_index(name='Product')


Engine Ready! Launching UI



## 4. Gradio Interface
Finally, we define the prediction logic to retrieve recommendations for a specific user and wrap it in a Gradio block to create our interactive web application.

In [10]:
def get_ui_recommendations(target_address, num_recommendations):
    if target_address not in user_item_matrix['Purchase Address'].values:
        return "Address not found.", "No recommendations available."

    target_customer_index = user_item_matrix[user_item_matrix['Purchase Address'] == target_address].index[0]

    # Fetch clean past purchases to display on the UI
    past_purchases_raw = set(prod_df[prod_df['Purchase Address'] == target_address]['Product'].values)

    similar_customers = similarity_matrix[target_customer_index].argsort()[::-1][1:(num_recommendations * 5) + 1]

    recommendations = []
    for customer_index in similar_customers:
        cust_addr = user_item_matrix.iloc[customer_index]['Purchase Address']
        cust_purchases = set(prod_df[prod_df['Purchase Address'] == cust_addr]['Product'].values)

        new_items = cust_purchases.difference(past_purchases_raw)
        for item in new_items:
            if item not in recommendations:
                recommendations.append(item)

        if len(recommendations) >= num_recommendations:
            break

    recs = recommendations[:num_recommendations]

    # Formatting for the Gradio Textbox
    formatted_past = "\n".join([f"✔️ {item}" for item in past_purchases_raw])
    if not recs:
        formatted_recs = "User has already purchased everything similar users bought!"
    else:
        formatted_recs = "\n".join([f"🚀 {item}" for item in recs])

    return formatted_past, formatted_recs

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🛒 E-Commerce Recommender System V2 (Feature Engineered)")
    gr.Markdown("Select a target customer to analyze their purchase history and generate collaborative-filtering recommendations based on Lookalike Audiences, Spending Habits, and Recency.")

    with gr.Row():
        with gr.Column(scale=1):
            user_dropdown = gr.Dropdown(
                choices=dropdown_choices,
                label="Target Customer Address",
                value=dropdown_choices[0] if dropdown_choices else None,
                info="Sample of high-frequency customers"
            )
            num_recs = gr.Slider(minimum=1, maximum=10, step=1, value=5, label="Number of Recommendations")
            submit_btn = gr.Button("Generate Recommendations", variant="primary")

        with gr.Column(scale=2):
            past_purchases_out = gr.Textbox(label="Customer's Past Purchases", lines=4, interactive=False)
            recommendations_out = gr.Textbox(label="AI Recommended Products", lines=5, interactive=False)

    submit_btn.click(
        fn=get_ui_recommendations,
        inputs=[user_dropdown, num_recs],
        outputs=[past_purchases_out, recommendations_out]
    )

demo.launch(share=True)

/tmp/ipykernel_18140/3870887057.py:36: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e133584175e95fe528.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
